In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import os
import requests
import time
from tqdm.auto import tqdm


In [2]:
import torch, transformers, accelerate
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)

torch 2.7.0
transformers 5.1.0
accelerate 1.12.0


In [3]:
ds = load_dataset("Tobi-Bueck/customer-support-tickets")

In [5]:
train_ds = ds["train"]

In [6]:
train_de_ds = train_ds.filter(lambda x: x["language"] == "de")

In [7]:
df = train_de_ds.to_pandas()

In [8]:
df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Anfrage nach detaillierten Angaben zur Systema...,"Sehr geehrtes Kundensupport-Team,\n\nich hoffe...",Vielen Dank für Ihre Anfrage. Wir stellen Ihne...,Request,Technical Support,low,de,51.0,Documentation,Feedback,IT,Tech Support,NaN,NaN,NaN,NaN
2,Anfrage zur Klärung der Auswirkungen eines Ser...,"Sehr geehrtes Kundendienstteam,\n\nich hoffe, ...","Vielen Dank, dass Sie uns bezüglich des kürzli...",Request,Service Outages and Maintenance,high,de,51.0,Disruption,Outage,Recovery,Support,NaN,NaN,NaN,NaN
3,Issue with SaaS Platform Functionality,"Sehr geehrtes Support-Team,\n\nich möchte Sie ...",Vielen Dank für Ihre Kontaktaufnahme bezüglich...,Incident,Product Support,medium,de,51.0,Bug,Performance,Disruption,Feature,NaN,NaN,NaN,NaN
4,Verbindungsstörung,"Sehr geehrtes Support-Team,\n\nich möchte ein ...","Vielen Dank, dass Sie sich wegen der Verbindun...",Problem,IT Support,high,de,51.0,Network,Disruption,Connectivity,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33499,Problem mit den digitalen Marketingkampagnen,"Sehr geehrte Kundensupport,\n\nich möchte mich...",Ich werde das Problem mit Ihren digitalen Mark...,Incident,Service Outages and Maintenance,medium,de,NaN,Technical,Customer,Bug,Outage,Maintenance,Documentation,NaN,NaN
33500,Frage an den Support,Ich schreibe in Bezug auf unerwartete Kosten a...,"Sehr geehrter <name>, ich möchte Sie über Ihre...",Incident,Billing and Payments,high,de,NaN,Billing,Payment,Refund,Customer,Issue,Documentation,NaN,NaN
33501,Bitten um Unterstützung bei der Integration,"Sehr geehrte Kundenservice, ich möchte die Int...","Sehr geehrte [Name], vielen Dank für Ihren Kon...",Change,Technical Support,medium,de,NaN,Integration,Feature,Documentation,Tech Support,NaN,NaN,NaN,NaN
33502,Hilfe bei digitalen Strategie-Problemen,Die Qualität unserer digitalen Strategie-Bearb...,Um den digitalen Strategie-Impuls zu überprüfe...,Incident,Product Support,high,de,NaN,Feedback,Performance,IT,Tech Support,NaN,NaN,NaN,NaN


In [ ]:
BASE = "http://127.0.0.1:5001"

def lt_translate(text, source="de", target="en"):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None
    r = requests.post(
        f"{BASE}/translate",
        json={"q": str(text), "source": source, "target": target, "format": "text"},
        timeout=60
    )
    r.raise_for_status()
    return r.json()["translatedText"]

def clean_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, (float, np.floating)) and float(v).is_integer():
        return int(v)
    return v

def make_english_ticket(row):
    row = row.to_dict() if hasattr(row, "to_dict") else dict(row)

    subject_en = lt_translate(row.get("subject"))
    body_en    = lt_translate(row.get("body"))
    answer_en  = lt_translate(row.get("answer"))

    out = {k: clean_value(v) for k, v in row.items()}
    out["subject"] = subject_en
    out["body"]    = body_en
    out["answer"]  = answer_en
    out["language"] = "en"
    return out

ticket_en = make_english_ticket(df.iloc[0])
ticket_en

{'subject': 'Essential safety incident',
 'body': 'Dear support team,\\n\\nich would like to report a serious safety incident that currently affects several components of our infrastructure. Affected devices include projectors, screens and storage solutions on cloud platforms. The reason for the assumption is that the incident is a potential breach of data related to a cyberattack, which means a significant risk for sensitive information and the ongoing operation of our organization. \\n\\nOur initial studies have shown unusual activities and deviations in the devices. Despite the implementation of our standardised recovery and containment measures, the threat has not yet been completely eliminated.',
 'answer': 'Thank you for reporting the critical safety incident and providing the overview of the devices concerned and the first measures taken. We recognise the urgency and severity of the situation, and we are all committed to addressing the case priority. For an immediate investigati

In [ ]:
CACHE_PATH = "df_tickets_translated_en.csv"

In [ ]:
BASE = "http://127.0.0.1:5001"
session = requests.Session()

def _is_missing(x):
    return x is None or (isinstance(x, float) and np.isnan(x))

def lt_translate_batch(texts, source="de", target="en", timeout=120):
    # texts: list[str]; keep missing as empty string and map back to None later
    payload = {"q": texts, "source": source, "target": target, "format": "text"}
    r = session.post(f"{BASE}/translate", json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()

    # LibreTranslate commonly returns {"translatedText": "..."} for single q,
    # and a list of {"translatedText": "..."} for list q. Handle both.
    if isinstance(data, list):
        return [item["translatedText"] for item in data]
    if isinstance(data, dict) and "translatedText" in data:
        # server treated it as single-item
        return [data["translatedText"]]
    raise ValueError(f"Unexpected response format: {type(data)}")

def translate_series(series: pd.Series, batch_size=16, sleep_s=0.0):
    vals = series.tolist()
    out = [None] * len(vals)

    # Build indices of non-missing items
    idxs = [i for i, v in enumerate(vals) if not _is_missing(v)]
    texts = [str(vals[i]) for i in idxs]

    # Batch translate
    pos = 0
    for start in tqdm(range(0, len(texts), batch_size)):
        chunk = texts[start:start + batch_size]
        try:
            translated = lt_translate_batch(chunk)
            if len(translated) != len(chunk):
                raise ValueError("Batch size mismatch")
        except Exception:
            # fallback: translate one-by-one to avoid losing progress
            translated = []
            for t in chunk:
                time.sleep(sleep_s)
                translated.append(lt_translate_batch([t])[0])

        # write back
        for j, tr in enumerate(translated):
            out[idxs[pos + j]] = tr
        pos += len(chunk)

        if sleep_s:
            time.sleep(sleep_s)

    return out

In [ ]:
TICKET_COL_ORDER = [
    "subject",
    "body",
    "answer",
    "type",
    "queue",
    "priority",
    "language",
    "version",
    "tag_1",
    "tag_2",
    "tag_3",
    "tag_4",
    "tag_5",
    "tag_6",
    "tag_7",
    "tag_8",
]
if os.path.exists(CACHE_PATH):
    df_en = pd.read_parquet(CACHE_PATH)
else:
    df_en = df.copy()

    # translate only these three fields
    df_en["subject"] = translate_series(df["subject"], batch_size=16)
    df_en["body"]    = translate_series(df["body"], batch_size=8)
    df_en["answer"]  = translate_series(df["answer"], batch_size=8)

    # set language to English
    df_en["language"] = "en"

    # reorder columns to match a "complete ticket" layout
    missing = [c for c in TICKET_COL_ORDER if c not in df_en.columns]
    if missing:
        raise ValueError(f"Missing expected columns in df_en: {missing}")

    df_en = df_en[TICKET_COL_ORDER]
    df_en.to_csv(CACHE_PATH, index=False)

100%|██████████| 2541/2541 [2:15:26<00:00,  3.20s/it]  


In [29]:
ds1 = pd.read_csv("df_tickets_translated_en.csv")

In [30]:
ds1.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Essential safety incident,"Dear support team,\n\nich would like to report...",Thank you for reporting the critical safety in...,Incident,Technical Support,high,en,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Request for detailed information on the platfo...,"Dear customer support team,\n\nich hope this m...",Thank you for your request. We provide you wit...,Request,Technical Support,low,en,51.0,Documentation,Feedback,IT,Tech Support,NaN,NaN,NaN,NaN
2,Request for clarification of the effects of a ...,"Dear customer service team,\n\nich hope this m...",Thank you for contacting us about the recent s...,Request,Service Outages and Maintenance,high,en,51.0,Disruption,Outage,Recovery,Support,NaN,NaN,NaN,NaN
3,Issue with SaaS Platform Functionality,"Dear support team,\n\nich would like to draw y...",Thank you for contacting us with our SaaS plat...,Incident,Product Support,medium,en,51.0,Bug,Performance,Disruption,Feature,NaN,NaN,NaN,NaN
4,Interference,"Dear support team,\n\nich would like to report...",Thank you for contacting us because of the con...,Problem,IT Support,high,en,51.0,Network,Disruption,Connectivity,Tech Support,NaN,NaN,NaN,NaN


In [32]:
ds1.shape

(33504, 16)

In [43]:
ds1.iloc[1].to_dict()

{'subject': 'Request for detailed information on the platform system architecture',
 'body': 'Dear customer support team,\\n\\nich hope this message will meet you. I contact you to get comprehensive information about the architecture of the platform. Understanding the underlying structure, components and their relationships is crucial to ensure smooth integration and to optimize the use of the services. \\n\\nI am particularly interested in details about the core modules of the platform, data streams, security measures, scalability features as well as available APIs and interfaces for adaptation. In addition, insights into the Technologiestack and the deployment environment would be very helpful.\\n\\n Access to this information allows the technical team to better plan and control the infrastructure processes.',
 'answer': 'Thank you for your request. We provide you with the available technical documentation. If necessary, please let us arrange a suitable appointment with our specialis